In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import sys; import os; import time; from datetime import timedelta

import numpy as np
import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
import xarray as xr

import pickle
import h5py

from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
def GetPlottingDirectory(plotFileName, plotType):
    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, 
                                             f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz")
    os.makedirs(specificPlottingDirectory, exist_ok=True)

    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName)

    return plottingFileName

def SaveFigure(fig,plotType, fileName):
    plotFileName = f"{fileName}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName, plotType)
    print(f"Saving figure to {plottingFileName}")
    fig.savefig(plottingFileName, dpi=300, bbox_inches='tight')

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","1_Domain_Profiles"))
from CLASSES_DomainProfiles import DomainProfiles_Class, DomainProfiles_DataLoading_Class

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Eulerian_CLTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
#############################
#SETUP

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=20
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=100
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
##############################################
#MODEL AND ALGORITHM NUMERICAL PARAMETERS

In [ ]:
dx=int(ModelData.xh[1]-ModelData.xh[0]) #grid resolution (in km)
dy=dx
xh = ModelData.xh - ModelData.xh[0]
yh = ModelData.yh - ModelData.yh[0]

In [ ]:
#########################################
#LOADING DATA FUNCTIONS

In [ ]:
#getting convergence xmax
def Get_AvgConvergence(t):

    timeString = ModelData.timeStrings[t]
    outputDataDirectory=os.path.normpath(os.path.join(DataManager.outputDataDirectory,"..","Eulerian_CLTracking"))
    Dictionary = TrackingAlgorithms_DataLoading_Class.LoadData(ModelData, DataManager, timeString,
                     dataName="Eulerian_CLTracking",outputDataDirectory=outputDataDirectory,printstatement=False)
    avgConvergence = Dictionary["avgConvergence"]
    return avgConvergence
    
def find_SBF_xmaxs():
    xmaxs=[]
    for t in tqdm(range(ModelData.Ntime)):
        if t == 0:
            xmaxs.append(np.nan)
        else:
            avgConvergence = Get_AvgConvergence(t)
            avgConvergence_max=np.nanmax(avgConvergence)
            xmax = np.where(avgConvergence==avgConvergence_max)[0][0]
            xmaxs.append(xmax)
    return xmaxs
xmaxs=find_SBF_xmaxs()

In [ ]:
def LoadTrackedData(t):
    timeString = ModelData.timeStrings[t]
    Dictionary = TrackingAlgorithms_DataLoading_Class.LoadData(ModelData,DataManager, timeString)
    print(Dictionary.keys())
    CP_SBF_Mask = Dictionary["CP_SBF_Mask"]
    CP_SBF_Mask_negation = Dictionary["CP_SBF_Mask_negation"]
    return CP_SBF_Mask,CP_SBF_Mask_negation

def GetVariableData(t,varName):
    timeString = ModelData.timeStrings[t]
    variableData = CallVariable(ModelData, DataManager, timeString, varName)
    return variableData

In [ ]:
#########################################
#PLOTTING FUNCTIONS

In [ ]:
def GetVariableData(t,varName):
    timeString = ModelData.timeStrings[t]
    variableData = CallVariable(ModelData, DataManager, timeString, varName)
    return variableData

def PlotQV(qv):
    plt.figure(figsize=(10, 4))
    plt.pcolormesh(qv, cmap='viridis')
    plt.xlim(left=oceanIndex, right=xh[-1]-50*kms)
    plt.colorbar(label="qv (kg/kg)")
    plt.axvline(xmaxs[t],color='k',linewidth=2)

def PlotQVPrime(qv_prime, axis=None, fig=None):
    v_limit = np.nanpercentile(np.abs(qv_prime), 95)
    norm = colors.TwoSlopeNorm(vmin=-v_limit, vcenter=0, vmax=v_limit)
    
    if axis is None:
        plt.figure(figsize=(10, 4))
        plt.pcolormesh(qv_prime, cmap='RdBu_r', norm=norm)
        plt.xlim(left=oceanIndex, right=xh[-1]-50*kms)
        plt.colorbar(label="qv' (kg/kg)")
        plt.axvline(xmaxs[t], color='k', linewidth=2)
    else:
        # FIX: Changed 'ax' to 'axis' to match your argument name
        pc = axis.pcolormesh(qv_prime, cmap='RdBu_r', norm=norm)
        axis.set_xlim(left=oceanIndex, right=xh[-1]-50*kms)
        # FIX: Only call fig.colorbar if fig is actually provided
        if fig is None:
            fig.colorbar(pc, ax=axis, label="qv' (kg/kg)")
        axis.axvline(xmaxs[t], color='k', linewidth=2)

def PlotQVPrime_Mask(qv_prime):
    data1 = qv_prime.copy(); data2 = qv_prime.copy()
    data1[~CP_SBF_Mask[z]] = np.nan
    # data2[~CP_SBF_Mask_negation[z]] = np.nan
    data2[CP_SBF_Mask[z]] = np.nan
    
    cmap = 'RdBu_r'
    fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

    v_limit = np.nanpercentile(np.abs(qv_prime), 98)
    v_min, v_max = -v_limit, v_limit 
    
    axes[0].pcolormesh(qv_prime, cmap=cmap, vmin=v_min, vmax=v_max)
    axes[0].set_title('No Mask')
    
    im = axes[1].pcolormesh(data1, cmap=cmap, vmin=v_min, vmax=v_max)
    axes[1].set_title('CL Mask')
    
    axes[2].pcolormesh(data2, cmap=cmap, vmin=v_min, vmax=v_max)
    axes[2].set_title('nonCL Mask')
    
    for ax in axes:
        ax.set_xlim(left=oceanIndex, right=xh[-1]-50*kms)
        # Ensure 't' and 'xmaxs' are available in scope
        ax.axvline(xmaxs[t], color='k', linewidth=2)
    
    # This colorbar now applies to the whole axes array
    fig.colorbar(im, ax=axes, orientation='vertical', label='QV')

z=0
kms=1000/ModelData.dx
xh=ModelData.xh-ModelData.xh[0]
oceanFraction=0.25
oceanIndex=int(np.round(xh[-1]*oceanFraction))

In [ ]:
#########################################
#LOADING DATA

In [ ]:
t = 60
# t=108
print(ModelData.timeStrings[t])
[CP_SBF_Mask,CP_SBF_Mask_negation] = LoadTrackedData(t)

qv = GetVariableData(t,"qv")[z]
qv[:,:oceanIndex]=np.nan
qv_prime = qv - np.nanmean(qv)

In [ ]:
#########################################
#PLOTTING

In [ ]:
PlotQVPrime_Mask(qv_prime)